In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import shutil
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# import os
# import shutil
# import random

# # === CONFIG ===
# source_dir = 'D:\\ML-research\\Datasets\\chest_xray_images'  # contains bacterial, viral, normal
# target_dir = 'D:\\ML-research\\Datasets\\chest_xray_images'   # will contain train/val/test folders
# split_ratios = (0.7, 0.15, 0.15)  # train, val, test

# # Ensure reproducibility
# random.seed(42)

# # Create target directories
# splits = ['train', 'val', 'test']
# classes = ['bacterial', 'viral', 'normal']

# for split in splits:
#     for cls in classes:
#         os.makedirs(os.path.join(target_dir, split, cls), exist_ok=True)

# # Split and copy files
# for cls in classes:
#     cls_path = os.path.join(source_dir, cls)
#     images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random.shuffle(images)

#     total = len(images)
#     train_end = int(split_ratios[0] * total)
#     val_end = train_end + int(split_ratios[1] * total)

#     split_files = {
#         'train': images[:train_end],
#         'val': images[train_end:val_end],
#         'test': images[val_end:]
#     }

#     for split, file_list in split_files.items():
#         for file in file_list:
#             src = os.path.join(cls_path, file)
#             dst = os.path.join(target_dir, split, cls, file)
#             shutil.copy2(src, dst)

# print("--> Dataset successfully split into train/val/test for all classes.")


--> Dataset successfully split into train/val/test for all classes.


In [ ]:
# # Set path to your original dataset
# original_train_path ="/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/train"

# # Target path for reorganized data
# output_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained"
# os.makedirs(output_path, exist_ok=True)

# # Create new class folders
# for label in ["normal", "bacterial", "viral"]:
#     os.makedirs(os.path.join(output_path, label), exist_ok=True)

# # Move NORMAL images to 'normal'
# normal_path = os.path.join(original_train_path, "NORMAL")
# for img in os.listdir(normal_path):
#     shutil.copy(os.path.join(normal_path, img), os.path.join(output_path, "normal"))

# # Move PNEUMONIA images into 'bacterial' or 'viral'
# pneumonia_path = os.path.join(original_train_path, "PNEUMONIA")
# for img in os.listdir(pneumonia_path):
#     img_lower = img.lower()
#     if "bacteria" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_path, "bacterial"))
#     elif "virus" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_path, "viral"))

# print("✅ Dataset split complete into 3 classes: normal, bacterial, viral")

In [ ]:
# # Set path to your original dataset
# original_test_path ="/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/test"

# # Target path for reorganized data
# output_test_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested"
# os.makedirs(output_test_path, exist_ok=True)

# # Create new class folders
# for label in ["normal", "bacterial", "viral"]:
#     os.makedirs(os.path.join(output_test_path, label), exist_ok=True)

# # Move NORMAL images to 'normal'
# normal_path = os.path.join(original_test_path, "NORMAL")
# for img in os.listdir(normal_path):
#     shutil.copy(os.path.join(normal_path, img), os.path.join(output_test_path, "normal"))

# # Move PNEUMONIA images into 'bacterial' or 'viral'
# pneumonia_path = os.path.join(original_test_path, "PNEUMONIA")
# for img in os.listdir(pneumonia_path):
#     img_lower = img.lower()
#     if "bacteria" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_test_path, "bacterial"))
#     elif "virus" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_test_path, "viral"))

# print("✅ Dataset split complete into 3 classes: normal, bacterial, viral")

In [ ]:
# # Set path to your original dataset
# original_val_path ="/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/val"

# # Target path for reorganized data
# output_val_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed"
# os.makedirs(output_val_path, exist_ok=True)

# # Create new class folders
# for label in ["normal", "bacterial", "viral"]:
#     os.makedirs(os.path.join(output_val_path, label), exist_ok=True)

# # Move NORMAL images to 'normal'
# normal_path = os.path.join(original_val_path, "NORMAL")
# for img in os.listdir(normal_path):
#     shutil.copy(os.path.join(normal_path, img), os.path.join(output_val_path, "normal"))

# # Move PNEUMONIA images into 'bacterial' or 'viral'
# pneumonia_path = os.path.join(original_val_path, "PNEUMONIA")
# for img in os.listdir(pneumonia_path):
#     img_lower = img.lower()
#     if "bacteria" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_val_path, "bacterial"))
#     elif "virus" in img_lower:
#         shutil.copy(os.path.join(pneumonia_path, img), os.path.join(output_val_path, "viral"))

# print("✅ Dataset split complete into 3 classes: normal, bacterial, viral")

In [ ]:

def clean_bad_images(folder_path):
    total_removed = 0
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.jpeg', '.jpg')):
                file_path = os.path.join(root, file)
                try:
                    with Image.open(file_path) as img:
                        img.verify()  # PIL will raise error if corrupt
                except Exception as e:
                    print(f"❌ Removed corrupted file: {file_path}")
                    os.remove(file_path)
                    total_removed += 1
    print(f"✅ Cleaning complete. Total removed: {total_removed}")

# Apply to all sets
clean_bad_images("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained")
clean_bad_images("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested")
clean_bad_images("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed")

In [ ]:
img_size = 224
batch_size = 32

train_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained"
test_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested"
val_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed"

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = val_datagen.flow_from_directory(
    test_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Load base model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False  # Freeze base

# Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()


In [ ]:
# Evaluate test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Predictions
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Classification report
print(classification_report(y_true, y_pred, target_names=test_generator.class_indices.keys()))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)
